# TRABALHO FINAL — Volis Case Study

Nome completo do estudante: José Alberto Alves de Melo

Dataset: volis_dataset.csv

Entrega: este notebook (.ipynb) devidamente preenchido e reprodutível

## Desafio de Machine Learning

In [ ]:
# ============================================================
#    (A) Classificação OU (B) Regressão OU (C) Clustering
# ============================================================

# Marque aqui qual desafio escolheu:
CHALLENGE = "C"  # "A" = Classificação | "B" = Regressão | "C" = Clustering

## 0) Importações e configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Para ter resultados reproduzíveis
RANDOM_STATE = 42

In [ ]:
# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [14, 8]

## 1) Carregamento dos dados

In [ ]:
DATA_PATH = "volis_dataset.csv"

df = pd.read_csv(DATA_PATH)

## 2) Visão geral do dataset

Compreender estrutura e distribuição inicial dos dados.

`df.shape`: Dimensão (linhas, colunas).

`df.sample()`: Mostra exemplos aleatórios.

`df.dtypes`: Tipos de variáveis.

`df.describe()`: Resumo estatístico (média, std, quartis).

In [ ]:
print("Dimensão (linhas, colunas):", df.shape)
display(df.sample(5, random_state=RANDOM_STATE))

print("\nTipos de dados:")
print(df.dtypes)

print("\nResumo estatístico (numéricas):")
display(df.describe())

## 3) Qualidade dos dados: missing values e duplicados

Valores em falta podem:
- Introduzir viés
- Reduzir performance de modelos

In [ ]:
# Missing values por coluna
print("\nValores em falta:")
display(df.isnull().sum())

# Percentual de missing por coluna
print("\nPercentual de valores em falta:")
display(df.isnull().mean() * 100)

# Linhas duplicadas
print("\nLinhas duplicadas:")
print(df.duplicated().sum())

# 4) Limpeza de Dados

In [ ]:
linhas_originais = df.shape[0]
print(f"N. linhas originais: {linhas_originais}")

# Como todos os valores em falta são inferiores a 5%, então vamos remover essas linhas
df_limpo = df.dropna()

linhas_depois = df_limpo.shape[0]
print(f"N. linhas após limpeza: {linhas_depois}")
print(f"Percentual a remover: {(1 - linhas_depois / linhas_originais):.0%}")

print("Percentagem muito elevada. Não remover. Necessário efetuar correções em vez de eliminar linhas.")

A percentagem das linhas removidas é de 16%, o que é muito elevado.
Tomada a decisão de não remover. Necessário efetuar correções em vez de eliminar linhas.

Remover dados só é aceitável quando:

- Percentagem é pequena (< 5%)
- Não causa viés

## Tratamento de variável "bl_private"

Esta variável categórica tem uma importância relevante na categorização das universidades.

Preencher valores em falta com valores imputados pode distorcer muito a realidade da análise.

Todas as linhas com valores em falta nesta coluna devem ser removidas.

In [ ]:
print(f"N. linhas originais: {linhas_originais}")

# Remover linhas cujo target é vazio/NaN
df = df.dropna(subset=["bl_private"])

# Converter variável 'bl_private' para numérica
df['bl_private'] = df['bl_private'].astype(int)

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

## Validar variáveis do tipo percentagem

- Garantir que variáveis percentuais estão entre 0–100
- Percentagens fora desse intervalo são inconsistências de dados

In [ ]:
# Analisar colunas percentagem
variaveis_percentagem = ['qt_top_10_percent','qt_top_25_percent','pc_faculty_with_phd','pc_faculty_with_terminal_degree','pc_alumni_donors','pc_graduation_rate']

for col in variaveis_percentagem:
    if col in df.columns:
        mask = (df[col] < 0) | (df[col] > 100)
        
        if mask.any():
            print(f"Coluna: {col}")
            display(df[mask])

Após análise dos dados anteriores foi decidido:
- Corrigir o valor 103 para 100 (pode ter sido um erro de arredondamento e a correção tem pouco impacto)
- Remover a linha cujo valor da percentagem é de 118 (valor demasiado elevado para correção e apenas 1 linha)

In [ ]:
# Corrigir percentagem
df['pc_faculty_with_phd'] = df['pc_faculty_with_phd'].replace(103, 100)
# Eliminar linha percentagem
df = df[((df['pc_graduation_rate'] < 118) | (df['pc_graduation_rate'].isnull()))]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

## Análise de Assimetria (Skewness)

Avaliar distribuição das variáveis.

- Skew $\gt$ 1 $\implies$ muito assimétrica
- Skew $\approx$ 0 $\implies$ simétrica

Distribuições muito assimétricas:

- Prejudicam distância Euclidiana
- Afetam K-Means

In [ ]:
variaveis_numericas = df.columns.tolist()
variaveis_numericas.remove('nm_college')

# análise da assimetria
skew_values = df[variaveis_numericas].skew().sort_values(ascending=False)

plt.figure()
skew_values.plot(kind="barh")
plt.axvline(0, color='black')
plt.title("Skewness por variável")
plt.show()

Separação de variáveis em 2 tipos:
- Variaveis Assimetricas
- Variaveis Simetricas

In [ ]:
# Separar variáveis consoante a sua simetria

variaveis_assimetricas = skew_values[skew_values > 1].index.tolist()

variaveis_simetricas = skew_values[skew_values <= 1].index.tolist()

## Analise de Correlação (Spearman)

Analisar dependência entre variáveis.

Dado o alto valor de assimetria (Skewness), foi decidido utilizar o método de *Spearman*, porque é mais robusto para dados não normais.

In [ ]:
# assimetria geral muito elevada - calcular correlação com método de 'spearman'
corr_spearman = df.corr(method='spearman', numeric_only=True)

plt.figure()
sns.heatmap(corr_spearman, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

Mediante o *heatmap* de correlação, foram criadas variáveis que contém as colunas correlacionadas entre si: ABS(correlação) >= 80%.

In [ ]:
# Atributos Correlacionados (>= 80%)

variaveis_corr_graduate = ['qt_applications_received','qt_applications_accepted','qt_students_enrolled','qt_undergraduate_students']
variaveis_corr_top_perc = ['qt_top_10_percent','qt_top_25_percent']
variaveis_corr_faculty = ['pc_faculty_with_phd','pc_faculty_with_terminal_degree']

## Tratamento de Outliers

Objetivo desta secção é corrigir ou eliminar outliers extremos (inferiores e superiores).

A presenção de outliers tem impacto relevante nos algorítmos. Em particular podem distorcer centróides e merges.

### Outliers extremos inferiores

Aqui queremos analisar a presença de valores extremos inferiores.

In [ ]:
# Analisar outliers extremos inferiores

variaveis_sem_private = variaveis_numericas.copy()
variaveis_sem_private.remove('bl_private')

# colunas com valores a 0/1
for col in variaveis_sem_private:
    print(f"Coluna: {col}: \n - Total valores inferiores a 1: {df[(df[col] <= 1)][col].count()} \n - Primeiro valor superior a 1 : {df[(df[col] > 1)][col].count()}")

- Dos dados da análise anterior, é seguro afirmar que existem valores inferiores a 1 (não passam 1 dezena por variável) que são outliers.
- A estratégia aqui é a de colocar esses valores como NaN para sofrerem imputação mais à frente.

In [ ]:
# Definir valores inferiores ou iguais a 1 como NaN
for col in variaveis_sem_private:
    df.loc[(df[col] <= 1), col] = np.nan

### Outliers extremos superiores

Aqui queremos analisar a presença de valores extremos superiores.

In [ ]:
# Analisar outliers extremos superiores
outliers_superiores = {}

for col in variaveis_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # limite superior clássico para identificar outliers
    upper = Q3 + 1.5 * IQR
    
    max_value = df[col].max()
    
    distance_upper = max(0, max_value - upper)
    
    outliers_superiores[col] = max(distance_upper, Q3)

pd.Series(outliers_superiores).sort_values(ascending=False)

- A partir dos dados anteriores, podemos verificar que existem outliers de grande dimensão.
- Para melhor efetuar a sua análise, foi decidido agrupar variáveis pela mesma dimensão do valor do *outlier* máximo.
- Desta análise resultou a criação de 3 grupos.

In [ ]:
# Variáveis com outliers mais expressivas

variaveis_outliers_1 = ['vl_expenditure_per_student', 'vl_tuition_outstate']
variaveis_outliers_2 = ['qt_applications_received','qt_undergraduate_students','vl_room_board','vl_personal_expenses','qt_applications_accepted','qt_postgraduate_students']
variaveis_outliers_3 = ['vl_books_cost', 'qt_students_enrolled']

### Análise do 1º grupo de Outliers

In [ ]:
# Análise de variaveis_outliers_1

plt.figure()
sns.boxplot(data=df[variaveis_outliers_1], color='lightblue')

- Nesta secção vamos, para cada variável com outlier, apresentar os valores que ficam acima do intervalo interquartil (Q3 + 1.5 * IQR).
- Para cada variável são apresentadas as linhas com valores superiores ao intervalor, bem como todas as outras colunas da mesma linha, de forma a poder encontrar algum padrão.

In [ ]:
for col in variaveis_outliers_1:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

- Da análise efetuada, foi decidido remover os outliers extremos.
- No caso da variável `vl_expenditure_per_student`, o outlier a remover tem o valor de 1.000.000 (1 milhão).
    - O valor seguinte mais alto tem o valor de 56.000.
    - Não tendo forma de saber se é um valor extremo real, achamos que é mais eficiente a sua remoção.
- No caso da variável `vl_tuition_outstate`, o outlier a remover tem o valor de 1.000.000 (1 milhão).
    - O valor seguinte mais alto tem o valor de 21.000.
    - Não tendo forma de saber se é um valor extremo real, achamos que é mais eficiente a sua remoção.

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['vl_expenditure_per_student'] < 1000000) | (df['vl_expenditure_per_student'].isnull())) &
        ((df['vl_tuition_outstate'] < 1000000) | (df['vl_tuition_outstate'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

- Análise do *boxplot* após as intervenções anteriores.

In [ ]:
# AVALIAR variaveis_outliers_1
plt.figure()
sns.boxplot(data=df[variaveis_outliers_1], color='lightblue')

### Análise do 2º grupo de Outliers

In [ ]:
# Análise de variaveis_outliers_2

plt.figure()
sns.boxplot(data=df[variaveis_outliers_2], color='lightblue')

- Análise do intervalo interquartil (Q3 + 1.5 * IQR).

In [ ]:
for col in variaveis_outliers_2:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

- No caso da variável `qt_applications_received`, o outlier mais extremo tem o valor de 100.000 (100 mil).
    - Analisado as colunas correlacionadas (variaveis_corr_graduate), foi decidio procurar a mediana para esta colunas com filtro pelos valores da colunas correlacionadas.
    - Foi encontrado um valor de 2000 que se parece enquadrar com as colunas correlacionadas.
    - Foi decidido atualizar estes valores de outliers para 2000.

In [ ]:
# Análise em colunas correlacionadas

filtro = df[(df['qt_students_enrolled'] == 500) & 
        (
          (df['qt_applications_accepted'] == 1500) | (df['qt_applications_accepted'].isna())
        )]

filtro_mediana = filtro['qt_applications_received'].median()

print(f"Valor mais comum no filtro: {filtro_mediana}")

In [ ]:
# Uma vez que colunas estão correlacionadas podemos fazer correção por regra lógica

df.loc[(df['qt_applications_received'] == 100000), 'qt_applications_received'] = 2000

- Da análise anterior foi detetada uma situação indêntica na variável `qt_applications_accepted`.
    - Foi seguida a mesma abordagem anterior e corrigida por regra lógica o valor desta variável.

In [ ]:
# Aproveitamos para corrigir também os NaN de 'qt_applications_accepted'

filtro = df[(df['qt_applications_received'] == 2000) & (df['qt_students_enrolled'] == 500) & (df['qt_applications_accepted'].notna())]

filtro_mediana = filtro['qt_applications_accepted'].median()

print(f"Valor mais comum no filtro: {filtro_mediana}")

In [ ]:
df.loc[(df['qt_applications_received'] == 20000) & (df['qt_students_enrolled'] == 500) & (df['qt_applications_accepted'].isna()), 'qt_applications_accepted'] = 1500

- Sobre os *outliers* das outras variáveis não foi encontrada nenhuma regra lógica.
    - Foi decidido remover as linhas destas variáveis, dado que a percentagem de remoção total das linhas fica pelos 3%.

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['qt_undergraduate_students'] < 50000) | (df['qt_undergraduate_students'].isnull())) &
        ((df['vl_room_board'] < 50000) | (df['vl_room_board'].isnull())) &
        ((df['vl_personal_expenses'] < 30000) | (df['vl_personal_expenses'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

- Análise do *boxplot* após as intervenções anteriores.

In [ ]:
# AVALIAR variaveis_outliers_2
plt.figure()
sns.boxplot(data=df[variaveis_outliers_2], color='lightblue')

### Análise do 3º grupo de Outliers

In [ ]:
# Análise de variaveis_outliers_3

plt.figure()
sns.boxplot(data=df[variaveis_outliers_3], color='lightblue')

- Análise do intervalo interquartil (Q3 + 1.5 * IQR).

In [ ]:
for col in variaveis_outliers_3:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

- Sobre os *outliers* destas variáveis também não foi encontrada nenhuma regra lógica.
    - Foi decidido remover as linhas destas variáveis, dado que a percentagem de remoção total das linhas fica pelos 3%.

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['vl_books_cost'] < 10000) | (df['vl_books_cost'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

- Análise do *boxplot* após as intervenções anteriores.

In [ ]:
# AVALIAR variaveis_outliers_3
plt.figure()
sns.boxplot(data=df[variaveis_outliers_3], color='lightblue')

## Análise de inconsistência entre variáveis correlacionadas

O objetivo desta secção é efetuar análise de inconsistência entre variáveis correlacionadas.

Foi procurado um padrão das inconsitências entre estas variáveis e aplicada uma regra lógica para a sua correção.

In [ ]:
# Análise de inconsistência entre variáveis correlacionadas
df_inconsistentes = df[
        (df["qt_applications_received"] < df["qt_applications_accepted"]) |
        (df["qt_applications_accepted"] < df["qt_students_enrolled"])
    ]

display(df_inconsistentes[variaveis_corr_graduate])

In [ ]:
#Correcao
df.loc[(df['qt_applications_received'] == 2000) & (df['qt_applications_accepted'] == 20000) & (df['qt_students_enrolled'] == 500), 'qt_applications_accepted'] = 2000
df.loc[(df['qt_applications_received'] == 2000) & (df['qt_applications_accepted'] == 1500) & (df['qt_students_enrolled'] == 10000), 'qt_students_enrolled'] = 1000

In [ ]:
# inconsistência
df_inconsistentes = df[(df["qt_top_10_percent"] > df["qt_top_25_percent"])]

print(f"Percentual de linhas inconsistentes: {(df_inconsistentes.shape[0] / df.shape[0]):.2%}")

display(df_inconsistentes)

In [ ]:
#Correção por regra lógica

df.loc[(df['qt_top_10_percent'] > 0) & (df['qt_top_10_percent'] < 100) & (df['qt_top_10_percent'] > df['qt_top_25_percent']), 'qt_top_25_percent'] = df['qt_top_10_percent']
df.loc[(df['qt_top_25_percent'] > 0) & (df['qt_top_25_percent'] < 100) & (df['qt_top_10_percent'] > df['qt_top_25_percent']), 'qt_top_10_percent'] = df['qt_top_25_percent']

In [ ]:
# inconsistência
df_inconsistentes = df[(df["pc_faculty_with_phd"] > df["pc_faculty_with_terminal_degree"])]

display(df_inconsistentes)

In [ ]:
#Correção por regra lógica

df.loc[(df['pc_faculty_with_phd'] > 0) & (df['pc_faculty_with_phd'] < 100) & (df['pc_faculty_with_phd'] > df['pc_faculty_with_terminal_degree']), 'pc_faculty_with_terminal_degree'] = df['pc_faculty_with_phd']

df.loc[(df['pc_faculty_with_terminal_degree'] > 0) & (df['pc_faculty_with_terminal_degree'] < 100) & (df['pc_faculty_with_phd'] > df['pc_faculty_with_terminal_degree']), 'pc_faculty_with_phd'] = df['pc_faculty_with_terminal_degree']

# 5) Imputação

Nesta secção pretende-se corrigir os dados em falta através da aplicação de algorítmos de imputação.

Na imputação foram aplicadas estratégias diferentes de média ou mediana para as variáveis simétricas e assimétricas, respetivamente.

In [ ]:
# IMPUTACAO

imputer_media = SimpleImputer(strategy='mean').set_output(transform="pandas")
df[variaveis_simetricas] = imputer_media.fit_transform(df[variaveis_simetricas])

imputer_mediana = SimpleImputer(strategy='median').set_output(transform="pandas")
df[variaveis_assimetricas] = imputer_mediana.fit_transform(df[variaveis_assimetricas])

print("Nulos após imputação:")
print(df.isnull().sum())

In [ ]:
variaveis_plot = ['qt_top_10_percent', 'qt_top_25_percent', 'pc_graduation_rate']

plt.figure()
sns.histplot(df[variaveis_plot], kde=True, bins=30, color='blue')

# 6) Transformação Log

Aqui pretendemos suavizar a curva das variáveis assimétricas, de forma a ter menos peso na aplicação do algorítmo de clustering.

In [ ]:
# Aplicar transformação logarítmica às variáveis assimetricas para suavizar a curva

df_transf = df[variaveis_numericas].copy()
df_transf[variaveis_assimetricas] = np.log1p(df[variaveis_assimetricas])

In [ ]:
plt.figure()
sns.histplot(df_transf[variaveis_plot], kde=True, bins=30, color='blue')

# 7) Normalização

Nesta secção pretendemos normalizar os valores de todas as variáveis numéricas de forma a ficarem na mesma escala.

Apesar de termos aplicado a transformação logarítmica, achamos por bem usar o método *RobustScaler* que é menos sensível a *outliers*.

In [ ]:
df_scaled = df_transf.copy();

scaler = RobustScaler()
df_scaled[variaveis_numericas] = scaler.fit_transform(df_transf[variaveis_numericas])

In [ ]:
sns.histplot(df_scaled[variaveis_plot], kde=True, bins=30, color='blue')

# 8) Seleção de Features

Para selecionar as variáveis a utilizar no algorítmo de clustering, decidimos avaliar a variância de cada variável e as respetivas correlações.

Tentamos selecionar as variáveis com maior variância uma vez que têm mais expressão na análise e evita o *overfitting* de variáveis com menor variância.
Decidimos também selecionar apenas uma variável entre as variáveis correlacionadas, de forma a evitar peso excessivo e correlações comuns.

O cálculo da variância foi apróximado à diferença entre os quartis (Q3 - Q1).

In [ ]:
print("Variância original:")
variancia_original = (df[variaveis_numericas].quantile(0.75) - df[variaveis_numericas].quantile(0.25)).sort_values(ascending=False)
print(variancia_original)

print("\nVariáveis correlacionadas")
print(f"Atributos de graduate: {variaveis_corr_graduate}")
print(f"Atributos de top percentagem: {variaveis_corr_top_perc}")
print(f"Atributos de faculty: {variaveis_corr_faculty}")

In [ ]:
# Selecionar a variáveis com maior variância e apenas uma entre as correlações existentes

variaveis_para_cluster = [
    'qt_undergraduate_students',
    'qt_top_25_percent',
    'pc_faculty_with_phd',
    'vl_tuition_outstate',
    'vl_expenditure_per_student',
    'pc_graduation_rate'
]

X_scaled = df_scaled[variaveis_para_cluster].to_numpy()

# 9) PCA

Aqui temos como objetivo a redução de dimensionalidade mantendo 95% da variância.

O PCA também ajuda a remover a redundância e melhora a eficiência de algorítmos baseados em distância, nomeadamente o K-Means.

O gráfico de PCA também ajuda a perceber a nuvem de pontos gerada.


In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled) # PCA sempre nos dados escalados!

print(f"O PCA selecionou automaticamente {pca.n_components_} componentes para atingir 95% de variância.")

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:,0], X_pca[:,1], X_pca[:,2])
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()

# 10) Método do Cotovelo

A partir do gráfico anterior de PCA, dá para perceber que não se encontram clusters visiveis a olho-nú.

Isto significa que os pontos forma uma nuvem de densidade única.

Assim sendo, aplicar um algorítmo baseado em distância deverá ser a melhor solução. Optámos pelo K-Means.

In [ ]:
sse = []
k_range = range(1, 11)

for k in k_range:
    kmeans_elbow = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans_elbow.fit(X_pca)
    sse.append(kmeans_elbow.inertia_)

# Plotar o gráfico do Cotovelo
plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o')
plt.title('Método do Cotovelo (Elbow Method)')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inércia (SSE)')
plt.grid(True)
plt.show()

# 11) K-Means

Para pesquisa de clusters, decidimos utilizar o K-Means para 3 *clusters*.

Analisando o método do cotovelo, o valor ideal de *clusters* seria entre 2 e 3.

In [ ]:
# Treinando com o k ideal
n_clusters = 3
kmeans_final = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
y_kmeans = kmeans_final.fit_predict(X_pca)

# Visualizando o resultado

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=y_kmeans, s=20, cmap='viridis')

# Plotar os centróides
centers = kmeans_final.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], c='red', s=200, alpha=0.7, label='Centróides')
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title(f"Clusterização Final (k={n_clusters})")
ax.legend()
plt.show()

- No gráfico é possível visualizar os 3 clusters. Também é visível que a separação entre os clusters não é a ideal.

# 12) Silhouette Score

Aqui pretendemos avaliar se a separação dos clusters é minimamente aceitável.

In [ ]:
score = silhouette_score(X_pca, y_kmeans)
print(f"Silhouette Score para k={n_clusters}: {score:.3f}")

# Interpretação rápida
if score > 0.5:
    print("Resultado: Clusters bem separados e definidos.")
elif score > 0.2:
    print("Resultado: Separação razoável.")
else:
    print("Resultado: Clusters sobrepostos ou mal definidos.")

# 13) Análise de Clusters

Nesta secção pretendemos analisar os *clusters* obtidos e extrair informação dessa análise.

In [ ]:
# Adiciona os clusters ao dataframe original

df['cluster'] = y_kmeans

In [ ]:
# Tamanho de cada cluster
# Isso mostra se temos clusters equilibrados ou um grupo dominante

cluster_size = df["cluster"].value_counts().sort_index()
print(cluster_size)

Relativamente ao cálculo das estatísticas, foi decidido aplicar o respetivo cálculo dividido pelas variáveis simétricas e assimétricas.

A ideia é usar a média sempre que nos referimos a variáveis simétricas e a mediana para variáveis assimétricas.

In [ ]:
# Estatísticas globais

media_global = df[variaveis_simetricas].mean()
mediana_global = df[variaveis_assimetricas].median()

estat_global = pd.concat([media_global, mediana_global])

In [ ]:
# Estatísticas por cluster / Centroids

media_cluster = df[variaveis_simetricas + ['cluster']].groupby('cluster').mean()
mediana_cluster = df[variaveis_assimetricas + ['cluster']].groupby('cluster').median()

estat_cluster = media_cluster.copy()

for col in mediana_cluster.columns:
    estat_cluster[col] = mediana_cluster[col]

No caso do cálculo do desvio padrão, foi decidido aplicar um tratamento especial para a variável `bl_private` dado que se trata de uma variável binária.

Em vez de usar o desvio padrão standard, cálculamos como: $\sigma = \sqrt{p(1-p)}$, em que $p$ é a proporção de 1's.

In [ ]:
# Comparar com média global

desvio_assimetricas = df[variaveis_assimetricas].std()
desvio_simetricas = df[variaveis_simetricas].std()
desvio_simetricas['bl_private'] = np.sqrt(df['bl_private'].mean()*(1-df['bl_private'].mean()))

# Padronização (Z-Score)
diff_desvio_simetricas = (media_cluster - media_global) / desvio_simetricas
diff_desvio_assimetricas = (mediana_cluster - mediana_global) / desvio_assimetricas

diff_norm = pd.concat([diff_desvio_simetricas, diff_desvio_assimetricas], axis=1)

In [ ]:
# variáveis que mais diferenciam cada cluster.

top_features = {}

for cluster in diff_norm.index:
    cluster_effects = diff_norm.loc[cluster]
    top = cluster_effects.sort_values(ascending=False)
    top_features[cluster] = top

#display(top_features)

### Visualização das estatísticas de cada cluster num *heatmap*

In [ ]:
plt.figure()
sns.heatmap(diff_norm, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5, linecolor="gray")

plt.title("Heatmap de Intensidade das Variáveis por Cluster", fontsize=16)
plt.ylabel("Cluster", fontsize=12)
plt.xlabel("Variáveis", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 14) Avaliação dos *Clusters*

Nesta secção vamos tentar obter significados dos valores apresentado no Heatmap de Intensidade das Variáveis por Cluster.

## Cluster 0

**Características acima da média**:

- Propinas elevadas (`vl_tuition_outstate` = 1.38)
- Elevada % de alunos no top 10% e 25% (`qt_top_10_percent` = 1.21, `qt_top_25_percent` = 1.04)
- Elevada despesa por aluno (`vl_expenditure_per_student` = 0.99)
- Alta qualificação do corpo docente (`pc_faculty_with_phd` = 0.99, `pc_faculty_with_terminal_degree` = 0.94)
- Alta taxa de graduação (`pc_graduation_rate` = 0.93)
- Maior percentagem de doações de alumni (`pc_alumni_donors` = 0.98)
- Tendência a serem privadas (`bl_private` > 0.50)

**Características abaixo da média**:

- Rácio aluno/docente mais baixo (`vl_student_faculty_ratio` = -0.66 $\implies$ Positivo em termos de qualidade)
- Menos despesas pessoais declaradas (`vl_personal_expenses` = -0.62)
- Menor dimensão em termos de número de estudantes (`qt_undergraduate_students` = -0.02, `qt_postgraduate_students` = -0.09)

**Interpretação**:

Este cluster representa universidades tendencialmente privadas, seletivas, com forte reputação académica e elevada qualidade, propinas altas e turmas mais pequenas.

## Cluster 1

**Características acima da média**:

- Grande número de estudantes (`qt_undergraduate_students` = 1.12, `qt_students_enrolled` = 1.10)
- Muitas candidaturas e admissões (`qt_applications_received` = 0.68, `qt_applications_accepted` = 0.71)
- Maior rácio aluno/docente (turmas maiores, `vl_student_faculty_ratio` = 0.72)
- Mais estudantes de pós-graduação (`qt_postgraduate_students` = 0.64)
- Despesas pessoais moderadamente acima da média (`vl_personal_expenses` = 0.44)

**Características abaixo da média**:

- Propinas mais baixas (`vl_tuition_outstate` = -0.94)
- Menor probabilidade de serem privadas (`bl_private` = -1.29)
- Taxa de graduação abaixo da média (`pc_graduation_rate` = -0.61)
- Menor percentagem de doações de antigos alunos ( `pc_alumni_donors` = -0.49)
- Menor despesa por estudante (`vl_expenditure_per_student` = -0.27)

**Interpretação**:

Este cluster representa universidades públicas de grande dimensão, menos seletivas, com propinas mais acessíveis, maior massificação e menor investimento por aluno.

## Cluster 2

**Características**:

- Valores próximos da média na maioria das variáveis
- Ligeiramente privadas (`bl_private` positivo, mas baixo)
- Ligeiramente abaixo da média em:
    - Qualificação do corpo docente (`pc_faculty_with_phd` = -0.60, `pc_faculty_with_terminal_degree` = -0.58)
    - Percentagem de alunos de topo (`qt_top_10_percent` = 0.00, `qt_top_25_percent` = -0.46)
    - Taxa de graduação (`pc_graduation_rate` = -0.18)
    - Número de candidaturas (`qt_applications_received` = -0.23)

**Interpretação**:

Este cluster parece representar instituições intermédias ou menos prestigiadas, nem muito grandes nem muito seletivas. Funcionam como grupo “médio” ou de menor diferenciação académica.

## Resumo Final

Cluster | Perfil dominante                | Tipo
:-----: | :------------------------------ | :--------------------
0       | Alta qualidade, seletivo, caro  | Privadas de elite
1       | Grande dimensão, acessível      | Públicas massificadas
2       | Intermédio / menos diferenciado | Pequenas ou regionais

## Ações de Gestão

- *Cluster* 0 - é de grande qualidade e deve ter uma gestão de topo.
- *Cluster* 1 - é referente a grandes universidades. Provavelmente baixando `vl_student_faculty_ratio` (turmas mais pequenas) iria ajudar na taxa de graduação.
- *Cluster* 2 - pequenas universidades. Aumentando a qualificação do corpo docente deveria aumentar a procura e trazer alunos dos top 10% e/ou 25%.

# 15) Conclusões e reflexão final

## 1) O que você descobriu no EDA que mudou sua visão do problema?

A análise exploratória de dados e a preparação de dados foi a parte mais demorada do projeto, em particular o tratamento de outliers e a correção de valores.

Sendo o objetivo evitar remover acima de 5% do total de linhas, existe necessidade de encontrar as inconsistências e tentar resolvê-las por regras de lógica.

Existem tantas particularidades no tratamento dos dados que, quanto maior for o dataset mais tempo será necessário despender a analisar e resolver cada situação.

## 2) Quais variáveis parecem mais ligadas ao sucesso académico (graduation_rate) e por quê?

Com base no mapa de correlação de *Spearman*, todas as variáveis com correlação absoluta acima de 0,3 influênciam `pc_graduation_rate` de forma significativa, nomeadamente:
- `bl_private` $\implies$ 0.33
- `qt_top_10_percent` $\implies$ 0.47
- `qt_top_25_percent` $\implies$ 0.47
- `vl_tuition_outstate` $\implies$ 0.57
- `vl_room_board` $\implies$ 0.36
- `pc_faculty_with_phd` $\implies$ 0.34
- `pc_faculty_with_terminal_degree` $\implies$ 0.32
- `pc_alumni_donors` $\implies$ 0.49
- `vl_expenditure_per_student` $\implies$ 0.42
- `vl_student_faculty_ratio` $\implies$ -0.32

Destas variáveis, diria que o facto de `vl_room_board` (custos de alojamento e alimentação) estar relacionado com o aumento da taxa de graduação é inesperado.

É muito provavel que existe correlação espúria em algumas destas variáveis individualmente, nomedamente `vl_room_board`, `pc_faculty_with_phd`, `pc_faculty_with_terminal_degree`.
No entanto, faz sentido que outras variáveis tenham relação de causalidade com a taxa de graduação, nomedamente `qt_top_10_percent` e `qt_top_25_percent`.

## 3) O desafio de ML escolhido ajudou a responder o quê?

O desafio de Clustering ajudou a segregar um conjunto de dados que à partida eram uma incógnita.

Apesar da homogeneidade dos dados a separação foi razoável (Silhouette Score para k=3: 0.330).

Analisando os dados foi possível encontrar relações entre as variáveis que refletem de forma real os vários tipos de universidades.

## 4) Limitações (dados, suposições, simplificações) e próximos passos realistas.

As principais limitações/desafios foram:

- Forma de tratar cada inconsistência que aparecia
- O cálculo da variância foi simplificado por Q3 - Q1
- Assumimos valores extremos de *outliers* como impossíveis (não tínhamos informação para confirmar)
- O cálculo da intensidade das variáveis por cluster também foi um desafio.

Os próximos passos podem passar por:
- Utilizar modelos supervisionados para prever taxa de graduação
- Realizar uma análise mais rigorosa de "feature importance"

## 5) Pergunta criada pelo grupo/estudante

### > Que características, pela negativa, caracterizam universidades de elite?

É do conhecimento comum que em universidades de elite as propinas são muito elevadas.

Mas uma característica que me surprendeu, foi o baixo número de despesas pessoais declaradas (`vl_personal_expenses` = -0.62).

Acredito que esta seja uma variável espúria, dado que são universidades privadas, com elevado valor de propinas (`vl_tuition_outstate` = 1.38).
